# Grover's Little Helper

**Complete | 400pts**
*Instructions | Submissions | Resources*

Grover's Algorithm is an oracle-based quantum algorithm for unstructured search in lists of $N$ items that finds—with high probability—the correct solutions using $O(\sqrt{N})$ queries of the list, whereas any classical algorithm would require $O(N)$ queries. This proven quadratic speedup makes Grover's algorithm one of the most promising quantum algorithms out there!

The workflow of Grover's algorithm consists of the following:

1. Prepare a uniform superposition of $n = \log_2(N)$ qubits:

$$\vert{}\psi\rangle = \frac{1}{\sqrt{N}} \sum_{x=0}^{N-1} \vert{}x\rangle.$$


2. Apply the oracle operator, which marks the sought-after states $\vert{}\omega\rangle$ with a minus sign.
3. Apply the diffusion operator to increase the amplitude of the states marked by the oracle.
4. Repeat 2 and 3 and measure.

The quantum circuit that represents Grover's algorithm for 2 qubits, with no additional ancillary qubits, is:

*(Image omitted)*

**NOTE:** Each computational basis state corresponds to one of the items in the list we are searching.

However, even masterpieces such as this algorithm have their own limitations! In this challenge, we will focus on one of the drawbacks of this algorithm with the goal to overcome it. In particular, for Grover's algorithm, it is especially challenging to find a solution when the number of solutions is half the total number of items, i.e. $N/2$. In this case, each iteration of Grover's operator (a combination of the oracle and diffusion operator), does not increase the amplitude of the solution states. You can check that yourself for one or two qubit circuits.

Can we still hope to find the solutions using Grover search? Of course we can!

To overcome this limitation we only need one auxiliary qubit! In fact, with one auxiliary qubit, we can design a circuit that requires applying the oracle and diffusion operator only once. Moreover, with one single shot, such circuit is able to provide one of the possible solutions!

In this challenge you are asked to design such a circuit! To avoid overcomplicating things, we will focus on the 2-qubit case, with four possible basis states:

$$\vert{}00\rangle, \vert{}01\rangle, \vert{}10\rangle, \vert{}11\rangle.$$

Two of them (you don't know which ones, but the oracle does 😉) are solutions. So, using ONLY one auxiliary qubit and ONLY one shot, you should be able to measure a quantum state that corresponds to one of the solutions.

## Challenge code

In the code below, you must complete this function:

* `circuit`: a QNode that returns a dictionary with the counts on the two first wires. The arguments of this function are:
* `generator (function)`: A function that initializes the quantum state to the uniform state
* `oracle (function)`: A function that marks the desired states
* `diffuser (function)`: A function that amplifies the states marked by the oracle



## Input

In this challenge there is no input. However, you can use the `generator`, `oracle` and `diffuser` functions, which can be called in the following way: `generator()`, `oracle()` and `diffuser(generator)`. Note that they all act over the first two qubits.

## Output

To judge this challenge, we will generate 1000 different oracles. Then, we will run your `circuit` function for each oracle to check that your predictions are correct. Therefore, in this case, there will be no public and private test cases.

All the best!

In [ ]:
import json
import pennylane as qp
import pennylane.numpy as np

dev = qp.device("default.qubit", wires=3)
@qp.set_shots(1)
@qp.qnode(dev)
def circuit(generator,oracle,diffuser):
    """Creates the circuit that implements Grover's algorithm with only one auxiliary qubit.

    Args:
        generator (function): A function that initializes the quantum state to the uniform state.
        oracle (function): A function that applies the oracle and marks the desired states
        diffuser (function): A function containing the operator that amplifies the states marked by the oracle. 
    
    Returns: 
        dict: Dictionary with the counts on the two first wires.      
    """


    # Put your code here #
    generator()
    diffuser(generator)

    qp.Hadamard(wires=2)
    qp.ctrl(oracle, control=2)()
    qp.Hadamard(wires=2)

    qp.CNOT(wires=[2, 0])
    qp.CNOT(wires=[2, 1])

    qp.PauliX(wires=0)
    qp.PauliX(wires=1)

    return qp.counts(wires=[0, 1])

# These functions are responsible for testing the solution.
def run(test_case_input: str) -> str:

    #useful functions
    @qp.prod
    def generator():
        # Creates the uniform initial quantum state
        qp.Hadamard(0)
        qp.Hadamard(1)
    def oracle():
        # Creates an oracle that marks two out of the four possible quantum states
        if random==0:
            qp.PauliZ(wires=0)
        elif random==1:
            qp.PauliX(wires=0)
            qp.PauliZ(wires=0)
            qp.PauliX(wires=0)
        elif random==2:
            qp.PauliZ(wires=1)
        else:
            qp.PauliX(wires=1)
            qp.PauliZ(wires=1)
            qp.PauliX(wires=1)            
    def diffuser(generator):
        # Creates the diffuser that reflects the state over the initial quantum state |𝜓⟩.
        qp.Reflection(generator()) 
    
    # private tests
    for _ in range(1000):
        random=np.random.randint(0,3)
        result=circuit(generator,oracle,diffuser)
        if random==0 and ("01" in result or "00" in result):
            return "Incorrect state"
        if random==1 and ("10" in result or "11" in result):
            return "Incorrect state"
        if random==2 and ("10" in result or "00" in result):
            return "Incorrect state"
        if random>=3 and ("01" in result or "11" in result):
            return "Incorrect state"
    return "Correct state"

def check(solution_output: str, expected_output: str) -> None:
    
    assert solution_output == "Correct state", "Not the correct quantum state."


# These are the public test cases
test_cases = [
    ('No input', 'No output')
]

# This will run the public test cases locally
for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)

    except Exception as exc:
        print(f"Runtime Error. {exc}")

    else:
        if message := check(output, expected_output):
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")

        else:
            print("Correct!")